# Amazon India Returns & Revenue Leakage — Phase 1A

This notebook performs initial inspection, cleaning, feature engineering, and business-question analysis for the Amazon India order dataset.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("D:/DA RESUME/amazon sales report/data/Amazon Sale Report.csv", low_memory=False)
df

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128970,128970,406-6001380-7673107,05-31-22,Shipped,Amazon,Amazon.in,Expedited,JNE3697,JNE3697-KR-XL,kurta,...,INR,517.00,HYDERABAD,TELANGANA,500013.0,IN,NaN,False,NaN,False
128971,128971,402-9551604-7544318,05-31-22,Shipped,Amazon,Amazon.in,Expedited,SET401,SET401-KR-NP-M,Set,...,INR,999.00,GURUGRAM,HARYANA,122004.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN,False
128972,128972,407-9547469-3152358,05-31-22,Shipped,Amazon,Amazon.in,Expedited,J0157,J0157-DR-XXL,Western Dress,...,INR,690.00,HYDERABAD,TELANGANA,500049.0,IN,NaN,False,NaN,False
128973,128973,402-6184140-0545956,05-31-22,Shipped,Amazon,Amazon.in,Expedited,J0012,J0012-SKD-XS,Set,...,INR,1199.00,Halol,Gujarat,389350.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN,False


INITIAL INSPECTION

In [3]:
print("Shape:", df.shape)
 
# Column names (check what you're working with)
print("\nColumns:\n", df.columns.tolist())
 
# Data types
print("\nData Types:\n", df.dtypes)
 
# First 5 rows
print("\nFirst 5 rows:\n", df.head())
 
# Null count per column
print("\nNull Count per Column:\n", df.isnull().sum())
 
# Null percentage per column
print("\nNull % per Column:\n", (df.isnull().sum() / len(df) * 100).round(2))
 
#Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (128975, 24)

Columns:
 ['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B', 'fulfilled-by', 'Unnamed: 22']

Data Types:
 index                   int64
Order ID               object
Date                   object
Status                 object
Fulfilment             object
Sales Channel          object
ship-service-level     object
Style                  object
SKU                    object
Category               object
Size                   object
ASIN                   object
Courier Status         object
Qty                     int64
currency               object
Amount                float64
ship-city              object
ship-state             object
ship-postal-code      float64
ship-country           object
promotion-ids          object
B2B                   

DATA CLEANING

In [4]:
#Strip whitespace from column names
df.columns=df.columns.str.strip()

In [5]:
#Removing whitespace from string
str_cols=['Status', 'Fulfilment','Sales Channel', 'ship-service-level','Category', 'Courier Status','currency','ship-city', 'ship-state','ship-country', 'fulfilled-by']
df[str_cols]=df[str_cols].apply(lambda x:x.str.strip())

In [6]:
#Check if any value still has leading/trailing space
for col in str_cols:
    has_space = df[col].dropna().str.contains(r'^\s|\s$', regex=True).any()
    print(f"{col}: whitespace remaining → {has_space}")

Status: whitespace remaining → False
Fulfilment: whitespace remaining → False
Sales Channel: whitespace remaining → False
ship-service-level: whitespace remaining → False
Category: whitespace remaining → False
Courier Status: whitespace remaining → False
currency: whitespace remaining → False
ship-city: whitespace remaining → False
ship-state: whitespace remaining → False
ship-country: whitespace remaining → False
fulfilled-by: whitespace remaining → False


In [7]:
#Convert 'Date' to datetime from object
df['Date']=pd.to_datetime(df['Date'],errors='coerce')


C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_4872\1620743315.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date']=pd.to_datetime(df['Date'],errors='coerce')


In [12]:
#Drop useless columns
cols_to_remove=['index','ASIN', 'ship-country', 'fulfilled-by', 'promotion-ids',
                'Sales Channel', 'Style', 'SKU', 'currency','B2B','ship-postal-code']
unnamed=[col for col in df.columns if 'Unnamed' in col]
df.drop(columns=cols_to_remove + unnamed,inplace=True,errors='ignore')

In [13]:
print('Remaining columns: ',df.columns)

Remaining columns:  Index(['Order ID', 'Date', 'Status', 'Fulfilment', 'ship-service-level',
       'Category', 'Size', 'Courier Status', 'Qty', 'Amount', 'ship-city',
       'ship-state'],
      dtype='object')


In [10]:
#Convert object data type to int
df['Qty']=df['Qty'].astype(int)

In [16]:
#Handling nulls
print((df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False))

Amount                6.04
Courier Status        5.33
ship-state            0.03
ship-city             0.03
Fulfilment            0.00
Status                0.00
Date                  0.00
Order ID              0.00
Size                  0.00
Category              0.00
ship-service-level    0.00
Qty                   0.00
dtype: float64


In [18]:
df['Amount']=df['Amount'].fillna(df['Amount'].mean())
df['Courier Status'].fillna('Unknown', inplace=True)
df['ship-city'].fillna('Unknown', inplace=True)
df['ship-state'].fillna('Unknown', inplace=True)
df['Size'].fillna('Unknown', inplace=True)


C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_4872\2228405389.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Courier Status'].fillna('Unknown', inplace=True)
C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_4872\2228405389.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

In [ ]:
#Cross check
df.isnull().sum()

Order ID              0
Date                  0
Status                0
Fulfilment            0
ship-service-level    0
Category              0
Size                  0
Courier Status        0
Qty                   0
Amount                0
ship-city             0
ship-state            0
dtype: int64

FEATURE ENGINEERING

In [20]:
df.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Size,Courier Status,Qty,Amount,ship-city,ship-state
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Standard,Set,S,Unknown,0,647.62,MUMBAI,MAHARASHTRA
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Standard,kurta,3XL,Shipped,1,406.00,BENGALURU,KARNATAKA
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Expedited,kurta,XL,Shipped,1,329.00,NAVI MUMBAI,MAHARASHTRA
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Standard,Western Dress,L,Unknown,0,753.33,PUDUCHERRY,PUDUCHERRY
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Expedited,Top,3XL,Shipped,1,574.00,CHENNAI,TAMIL NADU


In [ ]:
#Status column
df['Status'].unique()

array(['Cancelled', 'Shipped - Delivered to Buyer', 'Shipped',
       'Shipped - Returned to Seller', 'Shipped - Rejected by Buyer',
       'Shipped - Lost in Transit', 'Shipped - Out for Delivery',
       'Shipped - Returning to Seller', 'Shipped - Picked Up', 'Pending',
       'Pending - Waiting for Pick Up', 'Shipped - Damaged', 'Shipping'],
      dtype=object)

In [22]:
status_map = {
    'Shipped - Delivered to Buyer' : 'Delivered',
    'Shipped'                      : 'Shipped',
    'Shipped - Returned to Seller' : 'Returned',
    'Shipped - Rejected by Buyer'  : 'Rejected',
    'Shipped - Lost in Transit'    : 'Lost',
    'Shipped - Out for Delivery'   : 'Out for Delivery',
    'Shipped - Returning to Seller': 'Returning',
    'Shipped - Picked Up'          : 'Picked Up',
    'Shipped - Damaged'            : 'Damaged',
    'Pending'                      : 'Pending',
    'Pending - Waiting for Pick Up': 'Pending',
    'Shipping'                     : 'Shipping',
    'Cancelled'                    : 'Cancelled'
}

df['Status'] = df['Status'].map(status_map)

# Verify no status got missed
print(df['Status'].isnull().sum())
print(df['Status'].unique())

0
['Cancelled' 'Delivered' 'Shipped' 'Returned' 'Rejected' 'Lost'
 'Out for Delivery' 'Returning' 'Picked Up' 'Pending' 'Damaged' 'Shipping']


In [ ]:

df['Fulfilment'].unique()

array(['Merchant', 'Amazon'], dtype=object)

In [ ]:

df['ship-service-level'].unique()

array(['Standard', 'Expedited'], dtype=object)

In [26]:
df['Category'].unique()


array(['Set', 'kurta', 'Western Dress', 'Top', 'Ethnic Dress', 'Bottom',
       'Saree', 'Blouse', 'Dupatta'], dtype=object)

In [27]:
df['Size'].unique()	


array(['S', '3XL', 'XL', 'L', 'XXL', 'XS', '6XL', 'M', '4XL', '5XL',
       'Free'], dtype=object)

In [28]:
df['Courier Status'].unique()

array(['Unknown', 'Shipped', 'Cancelled', 'Unshipped'], dtype=object)

In [29]:
#Final check
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.head(3))

(128975, 12)
Order ID                      object
Date                  datetime64[ns]
Status                        object
Fulfilment                    object
ship-service-level            object
Category                      object
Size                          object
Courier Status                object
Qty                            int64
Amount                       float64
ship-city                     object
ship-state                    object
dtype: object
Order ID              0
Date                  0
Status                0
Fulfilment            0
ship-service-level    0
Category              0
Size                  0
Courier Status        0
Qty                   0
Amount                0
ship-city             0
ship-state            0
dtype: int64
              Order ID       Date     Status Fulfilment ship-service-level  \
0  405-8078784-5731545 2022-04-30  Cancelled   Merchant           Standard   
1  171-9198151-1101146 2022-04-30  Delivered   Merchant           Standa

In [30]:
#Save cleaned dataset
df.to_csv('D:/DA RESUME/amazon sales report/data/cleaned_amazon_dataset.csv',index=False)